# Cleaned Pinecone + LangChain Notebook
This notebook provides a minimal, consistent pipeline: install deps, load PDFs, split text, create embeddings with Hugging Face, upsert into Pinecone, create a retriever and a single QA query.

Notes: do NOT hardcode API keys — set `PINECONE_API_KEY`, `PINECONE_API_ENV`, and `HUGGINGFACEHUB_API_TOKEN` in your environment or a `.env` file.

In [1]:
# Install required packages (run once)
# Uncomment when needed in a fresh environment
# !pip install 
# pinecone-client huggingface_hub pypdf tiktoken python-dotenv

In [2]:
# Imports and environment
import os
from dotenv import load_dotenv
from langchain.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEndpoint
from pinecone import Pinecone

load_dotenv()  # loads .env into os.environ if present
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
PINECONE_API_ENV = os.getenv('PINECONE_API_ENV')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')
# Provide a safe default for the index name so later cells won't error if Pinecone isn't initialized
index_name = os.getenv('PINECONE_INDEX_NAME', 'test')
if not PINECONE_API_KEY or not HUGGINGFACEHUB_API_TOKEN:
    print('Warning: Make sure PINECONE_API_KEY and HUGGINGFACEHUB_API_TOKEN are set in your environment.')

c:\Users\Admin\Desktop\genai project1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load PDFs from `pdfs/` directory and split into chunks
loader = PyPDFDirectoryLoader('pdfs')
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
text_chunks = text_splitter.split_documents(docs)
print(f'Loaded {len(docs)} documents -> {len(text_chunks)} chunks')

Loaded 15 documents -> 90 chunks


In [4]:
# Initialize Hugging Face embeddings (local or hub)
# Uses `HUGGINGFACEHUB_API_TOKEN` from env when necessary
embedding = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
# Quick sanity: embed a short query (optional)
_ = embedding.embed_query('test')

C:\Users\Admin\AppData\Local\Temp\ipykernel_14056\2123723025.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')


In [5]:
# Initialize Pinecone client (modern client)
if not PINECONE_API_KEY or not PINECONE_API_ENV:
    print('Skipping Pinecone init — missing env vars')
else:
    pc = Pinecone(api_key=PINECONE_API_KEY, environment=PINECONE_API_ENV)
    index_name = os.getenv('PINECONE_INDEX_NAME', 'test')
    print('Pinecone client initialized; index_name=', index_name)

Pinecone client initialized; index_name= test


In [6]:
# Create or reuse a Pinecone-backed vectorstore and upsert embeddings
# This uses the langchain_pinecone wrapper
try:
    vectorstore = PineconeVectorStore.from_documents(
        documents=text_chunks,
        embedding=embedding,
        index_name=index_name
    )
    print('Vectorstore ready')
except Exception as e:
    print('Error creating/upserting vectorstore:', e)

Vectorstore ready


In [7]:
# Build retriever and QA chain using a small, stable HF endpoint or your preferred LLM
from langchain_ollama import OllamaLLM


llm = OllamaLLM(model='phi3:3.8b')  # Replace with your HF endpoint or local LLM
retriever = vectorstore.as_retriever()
qa = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever)
print('QA chain ready')

QA chain ready


In [8]:
# Run a single query with simple error handling
query = "What is the document about?"
try:
    # Using the newer invoke method is recommended, 
    # but run() still works in older LangChain versions
    answer = qa.invoke(query)
    print("Answer:\n", answer["result"])
except Exception as e:
    print("Error running QA:", e)

Answer:
 The documents discuss various topics related to natural language processing (NLP). Document [25] by Marcus et al., introduces The Penn Treebank, a large annotated corpus of English for computational linguistics research. This corpora includes syntactically and semantically annotated sentences which provide valuable data sets in the field of NLP.

Document [26], on the other hand, is focused on self-training methods used by McClosky et al., specifically discussing how these techniques can be effectively applied to parsing tasks within computational linguistics. Parsing refers to determining the syntactic structure or grammatical relationship of words in a sentence - essentially breaking down sentences into constituents like noun phrases, verb phrases etc.

Finally, document [27] by Uszkoreit et al., discusses attention mechanisms and their decomposability, which are essential concepts for developing neural network architectures used in various language understanding tasks such 

# Cleanup notes
- This consolidated notebook removes duplicate imports, repeated setups, and inline API keys.
- If you want, I can replace the original `Pineconevectordb.ipynb` with this cleaned version, or keep both files.

In [9]:
docs = docssearch.similarity_search(query)
print(docs)

NameError: name 'docssearch' is not defined

In [10]:
import sys
while True:
    user_input = input("Enter your question (or 'exit' to quit): ")
    if user_input.lower() == 'exit':
        print("Goodbye!")
        sys.exit()
    if user_input == '':
        continue
    result = qa.invoke({'query': user_input})
    print(f"Answer: {result['result']}\n")

Answer: Self-attention layers can be more efficient in terms of computational complexity compared to both recurrent (like LSTM, GRU) and conventional Convolutional Neural Networks (CNN), especially on tasks that require long-range dependencies modeling. This is due to self-attention's ability to capture relationships between distant elements within the input sequence by weighing their interactions based on relevance without being constrained by a fixed receptive field, as CNN does or recurrence in RNN and LSTM models are limited by.

Consequently, Self Attention (also known as Scaled Dot-Product Attention) has become increasingly popular for various tasks including machine translation, text summarization, and other natural language processing applications due to its strength of capturing global dependencies irrespective of the input size or sequence length - something that RNNs struggle with because they process sequences sequentially.

Self attention was a key component in models like

SystemExit: 

c:\Users\Admin\Desktop\genai project1\.venv\lib\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
